# RVL-CDIP Document Classifier — Production Training Notebook

Fine-tunes **ConvNeXt Tiny** on the RVL-CDIP 16-class document layout dataset.  
Produces the four artifacts that ship to the production repo:

| Artifact | Path in repo |
|---|---|
| Trained weights | `app/classifier/models/classifier.pt` |
| Model card + SHA-256 | `app/classifier/models/model_card.json` |
| 50 golden TIFFs | `app/classifier/eval/golden_images/` |
| Expected outputs | `app/classifier/eval/golden_expected.json` |

**Runtime:** Google Colab T4 GPU (free tier). Full training ≈ 90–120 min.  
**Run cells top-to-bottom in order. Do not skip cells.**

## Cell 1 — Install & imports

Pin versions to match `requirements.txt` in the production repo.

In [2]:
# ── 1. Install extra packages not present in Colab by default ─────────────────
!pip install -q \
    datasets==2.20.0 \
    torchmetrics==1.4.0 \
    matplotlib==3.9.0 \
    seaborn==0.13.2 \
    Pillow==10.4.0

# ── 2. Standard library ───────────────────────────────────────────────────────
import hashlib
import json
import os
import random
import shutil
import time
from pathlib import Path
from typing import Dict, List, Tuple

# ── 3. Numerics / vision ──────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
# torch.amp (replaces deprecated torch.cuda.amp)
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image

# ── 4. Metrics & plotting ─────────────────────────────────────────────────────
from torchmetrics import Accuracy
from torchmetrics.classification import MulticlassConfusionMatrix
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── 5. Verify GPU ─────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch   : {torch.__version__}")
print(f"device  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Training will be very slow.")

torch   : 2.10.0+cu128
device  : cuda
GPU     : Tesla T4
VRAM    : 15.6 GB


## Cell 2 — Global config

One place for every hyperparameter. Change values here only.

In [3]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Deterministic ops (small speed cost — required for golden set replay)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Dataset ───────────────────────────────────────────────────────────────────
# RVL-CDIP root after extracting the archive in Colab
DATA_ROOT = Path("/content/rvl-cdip")
LABELS_DIR = DATA_ROOT / "labels"   # train.txt / val.txt / test.txt
IMAGES_DIR = DATA_ROOT / "images"   # images/xxx/yyy/zzz.tif

CLASSES = [
    "letter", "form", "email", "handwritten", "advertisement",
    "scientific_report", "scientific_publication", "specification",
    "file_folder", "news_article", "budget", "invoice",
    "presentation", "questionnaire", "resume", "memo",
]
NUM_CLASSES = len(CLASSES)  # 16

# ── Model ─────────────────────────────────────────────────────────────────────
BACKBONE      = "convnext_tiny"          # or "convnext_small" for +1% accuracy at 2× cost
WEIGHTS_ENUM  = "IMAGENET1K_V1"          # exact torchvision weights identifier
IMG_SIZE      = 224                      # px — keeps fine document structure visible , set to 384 for better accuracy
CROP_SIZE     = 224                      # center-crop to same size , also 384

# ── Training schedule ─────────────────────────────────────────────────────────
BATCH_SIZE       = 64    # reduce to 16 if OOM on free Colab / was 32 — fewer steps per epoch
NUM_WORKERS      = 0    # must be 0 for HF streaming — avoids multiprocess conflicts

PROBE_EPOCHS     = 5     # frozen backbone — head-only training
FINETUNE_EPOCHS  = 8    # partial unfreeze — backbone stages 6+7 + head / was 15 — enough for 15% subset
TOTAL_EPOCHS     = PROBE_EPOCHS + FINETUNE_EPOCHS

PROBE_LR         = 1e-3  # head during linear probe
HEAD_LR          = 5e-4  # head during fine-tune
BACKBONE_LR      = 1e-4  # unfrozen backbone stages during fine-tune

LABEL_SMOOTHING  = 0.1   # makes confidence honest
WEIGHT_DECAY     = 1e-4
GRAD_CLIP        = 1.0
EARLY_STOP_PAT   = 4     # val accuracy patience

# ── Calibration & thresholds ──────────────────────────────────────────────────
TEMP_INIT        = 1.5   # starting point for temperature scaling search
REVIEWER_THRESH  = 0.7   # predictions below this are relabel-eligible

# ── Output paths (mirrored to Google Drive) ───────────────────────────────────
DRIVE_ROOT   = Path("/content/drive/MyDrive/rvlcdip_artifacts")
OUT_WEIGHTS  = DRIVE_ROOT / "classifier.pt"
OUT_CARD     = DRIVE_ROOT / "model_card.json"
OUT_GOLDEN   = DRIVE_ROOT / "golden_images"
OUT_EXPECTED = DRIVE_ROOT / "golden_expected.json"

GOLDEN_N     = 50        # total golden images
GOLDEN_EASY  = 2         # top-confidence picks per class  (2 × 16 = 32)
GOLDEN_HARD  = 18        # ambiguous cases picked from confusion matrix

# ── Accuracy threshold that production startup enforces ───────────────────────
MIN_TOP1_THRESHOLD = 0.82   # lower for 15% subset; raise to 0.90 for full dataset

print("Config loaded.")
print(f"  backbone : {BACKBONE} / {WEIGHTS_ENUM}")
print(f"  img size : {IMG_SIZE}px")
print(f"  epochs   : {PROBE_EPOCHS} probe + {FINETUNE_EPOCHS} finetune = {TOTAL_EPOCHS}")
print(f"  batch    : {BATCH_SIZE}")
print(f"  device   : {DEVICE}")

Config loaded.
  backbone : convnext_tiny / IMAGENET1K_V1
  img size : 224px
  epochs   : 5 probe + 8 finetune = 13
  batch    : 64
  device   : cuda


In [4]:
import os, shutil
from pathlib import Path

# Check disk
total, used, free = shutil.disk_usage("/content")
print(f"Disk free : {free/1e9:.1f} GB")
print(f"Disk used : {used/1e9:.1f} GB")

# Check HF cache contents
hf_cache = Path(os.path.expanduser("~/.cache/huggingface"))
if hf_cache.exists():
    for item in sorted(hf_cache.rglob("*")):
        if item.is_dir():
            try:
                size = sum(f.stat().st_size for f in item.rglob("*") if f.is_file())
                if size > 1e6:  # only show items > 1 MB
                    print(f"  {str(item.relative_to(hf_cache)):<60s} {size/1e9:.2f} GB")
            except:
                pass
else:
    print("No HF cache found.")

Disk free : 74.8 GB
Disk used : 46.1 GB


In [5]:
import os, shutil

hf_cache = os.path.expanduser("~/.cache/huggingface/datasets")
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache)
    print("Cleared.")

import shutil
total, used, free = shutil.disk_usage("/content")
print(f"Disk free : {free/1e9:.1f} GB")

Cleared.
Disk free : 74.8 GB


## Cell 3 — Download & mount dataset

RVL-CDIP lives at adamharley.com. Download once and cache in Drive.

In [6]:
from google.colab import drive, output
import random, shutil, os
import threading, time

# Keep the Colab runtime alive without relying on brittle DOM selectors
def _keep_alive():
    while True:
        try:
            output.eval_js('0')  # lightweight ping — keeps the runtime active
        except Exception:
            pass
        time.sleep(30)

_ka_thread = threading.Thread(target=_keep_alive, daemon=True)
_ka_thread.start()
print("Keep-alive thread started.")

drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
OUT_GOLDEN.mkdir(parents=True, exist_ok=True)

# Clear any broken processed cache — keeps the 38 GB raw downloads
processed = os.path.expanduser(
    "~/.cache/huggingface/datasets/aharley___rvl_cdip"
)
if os.path.exists(processed):
    shutil.rmtree(processed)
    print("Cleared failed processed cache.")

total, used, free = shutil.disk_usage("/content")
print(f"Disk free : {free/1e9:.1f} GB")

# ── streaming=True — reads from local cache, writes NOTHING to disk ───────────
from datasets import load_dataset

print("Loading in streaming mode...")
hf_train = load_dataset("aharley/rvl_cdip", split="train",
                        streaming=True, trust_remote_code=True)
hf_val   = load_dataset("aharley/rvl_cdip", split="validation",
                        streaming=True, trust_remote_code=True)
hf_test  = load_dataset("aharley/rvl_cdip", split="test",
                        streaming=True, trust_remote_code=True)

# Streaming datasets don't support .select() so we pre-pick random indices
SUBSET_FRACTION = 0.15
TRAIN_TOTAL, VAL_TOTAL, TEST_TOTAL = 320000, 40000, 40000

random.seed(SEED)
train_indices = set(random.sample(range(TRAIN_TOTAL),
                    int(TRAIN_TOTAL * SUBSET_FRACTION)))
val_indices   = set(random.sample(range(VAL_TOTAL),
                    int(VAL_TOTAL   * SUBSET_FRACTION)))
test_indices  = set(random.sample(range(TEST_TOTAL),
                    int(TEST_TOTAL  * SUBSET_FRACTION)))

print(f"Indices selected:")
print(f"  train : {len(train_indices):,} of {TRAIN_TOTAL:,}")
print(f"  val   : {len(val_indices):,} of {VAL_TOTAL:,}")
print(f"  test  : {len(test_indices):,} of {TEST_TOTAL:,}")
print("Done. No data written to disk.")

<IPython.core.display.Javascript object>

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Disk free : 74.8 GB
Loading in streaming mode...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Indices selected:
  train : 48,000 of 320,000
  val   : 6,000 of 40,000
  test  : 6,000 of 40,000
Done. No data written to disk.


## Cell 4 — Dataset class & transforms

Grayscale TIFFs → 3-channel RGB tensors with ImageNet normalization.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize(IMG_SIZE + 32),
    T.RandomCrop(CROP_SIZE),
    # RandomHorizontalFlip removed — flipping text documents is semantically unnatural
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.CenterCrop(CROP_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class StreamingSubsetDataset(Dataset):
    """
    Scans the HF streaming dataset once and loads only the
    selected indices into RAM. One-time cost ~5-8 min.
    After that training reads from RAM — fast and no disk needed.
    """

    def __init__(self, hf_stream, keep_indices, transform=None, name=""):
        self.transform = transform
        self.data      = []   # list of (PIL image, int label)
        total          = len(keep_indices)

        print(f"  [{name}] scanning stream, keeping {total:,} samples...")
        for i, item in enumerate(hf_stream):
            if i in keep_indices:
                self.data.append((item["image"], item["label"]))
            if (i + 1) % 20000 == 0:
                print(f"    scanned {i+1:,} — "
                      f"kept {len(self.data):,}/{total:,}", end="\r")
            if len(self.data) == total:
                break

        self.samples = [(i, self.data[i][1]) for i in range(len(self.data))]
        print(f"  [{name}] done — {len(self.data):,} samples ready.      ")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image, label = self.data[idx]
        image = image.convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


print("Building datasets — one-time scan, ~5-8 min total\n")
train_ds = StreamingSubsetDataset(
    hf_train, train_indices, train_transform, "train")
val_ds   = StreamingSubsetDataset(
    hf_val,   val_indices,   eval_transform,  "val")
test_ds  = StreamingSubsetDataset(
    hf_test,  test_indices,  eval_transform,  "test")

print(f"\ntrain_ds : {len(train_ds):,}")
print(f"val_ds   : {len(val_ds):,}")
print(f"test_ds  : {len(test_ds):,}")

# Weighted sampler
train_labels   = [s[1] for s in train_ds.samples]
class_counts   = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights  = 1.0 / (class_counts + 1e-6)
sample_weights = torch.tensor(
    [class_weights[l] for l in train_labels], dtype=torch.float)
sampler = WeightedRandomSampler(
    sample_weights, num_samples=len(train_ds), replacement=True)

# Guard: every class must have at least one sample in the subset
assert (class_counts == 0).sum() == 0, (
    f"Classes with zero samples in subset: {list(np.where(class_counts == 0)[0])}. "
    "Re-run Cell 3 with a larger SUBSET_FRACTION or a different SEED."
)

# num_workers=2 is safe now — data lives in RAM, not network
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=2,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE * 2,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Steps per train epoch : {len(train_loader)}")
print(f"Steps per val epoch   : {len(val_loader)}")

Building datasets — one-time scan, ~5-8 min total

  [train] scanning stream, keeping 48,000 samples...


### Visual check: sample images from each class

Verifies the dataset loaded correctly and augmentation looks sensible.

In [ ]:
import matplotlib.pyplot as plt

# Show one sample per class from the training set
# NOTE: HF streaming datasets don't support integer indexing — iterate instead
print("Loading one sample per class for visual check...")
seen = {}
for item in hf_train:
    label = item["label"]
    if label not in seen:
        seen[label] = item["image"].convert("RGB")
    if len(seen) == NUM_CLASSES:
        break

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle("One sample per class (no augmentation)", fontsize=14, fontweight="bold")

for cls_idx, ax in enumerate(axes.flat):
    if cls_idx in seen:
        ax.imshow(seen[cls_idx], cmap="gray")
    ax.set_title(CLASSES[cls_idx], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "sample_grid.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved sample_grid.png")


## Cell 5 — Build model

ConvNeXt Tiny with pretrained ImageNet weights. New 16-class head replaces the 1000-class original.

In [ ]:
def build_model() -> nn.Module:
    """
    Loads ConvNeXt Tiny with IMAGENET1K_V1 weights.
    Replaces the classifier head with Linear(768, NUM_CLASSES).
    Returns model moved to DEVICE with backbone frozen.

    Architecture:
      ConvNeXt Tiny backbone (features[0..7]) → AdaptiveAvgPool → flatten
      → LayerNorm → Linear(768, 16)
    """
    weights = models.ConvNeXt_Tiny_Weights[WEIGHTS_ENUM]
    model   = models.convnext_tiny(weights=weights)

    # The original classifier is: [LayerNorm, Flatten, Linear(768, 1000)]
    # We replace only the final Linear to keep LayerNorm pretrained.
    in_features = model.classifier[2].in_features  # 768
    model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)

    return model.to(DEVICE)


def freeze_backbone(model: nn.Module) -> None:
    """Freeze all parameters except the classifier head."""
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False


def unfreeze_last_stages(model: nn.Module) -> None:
    """
    Unfreeze ConvNeXt stages 6 and 7 (deepest feature stages).
    Stages 0-5 stay frozen — their low-level features need no adaptation.
    """
    for name, param in model.named_parameters():
        if "features.6" in name or "features.7" in name or "classifier" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False


def count_trainable(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model = build_model()
freeze_backbone(model)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = count_trainable(model)
print(f"Total params     : {total_params:>12,}")
print(f"Trainable (probe): {trainable_params:>12,}  ({100*trainable_params/total_params:.1f}%)")
print(f"\nClassifier head  : {model.classifier}")

## Cell 6 — Training helpers (loss, optimizer, scheduler, early stopping)

In [ ]:
# ── Loss ──────────────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# ── Metrics (torchmetrics — GPU-aware) ────────────────────────────────────────
top1_metric = Accuracy(task="multiclass", num_classes=NUM_CLASSES, top_k=1).to(DEVICE)
top5_metric = Accuracy(task="multiclass", num_classes=NUM_CLASSES, top_k=5).to(DEVICE)


def make_probe_optimizer(model: nn.Module) -> torch.optim.AdamW:
    """AdamW on head parameters only."""
    head_params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(head_params, lr=PROBE_LR, weight_decay=WEIGHT_DECAY)


def make_finetune_optimizer(model: nn.Module) -> torch.optim.AdamW:
    """Differential LR: backbone stages 5× lower than head."""
    backbone_params, head_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "classifier" in name:
            head_params.append(param)
        else:
            backbone_params.append(param)
    return torch.optim.AdamW([
        {"params": head_params,     "lr": HEAD_LR},
        {"params": backbone_params, "lr": BACKBONE_LR},
    ], weight_decay=WEIGHT_DECAY)


class EarlyStopping:
    """Halts training when validation top-1 stops improving."""

    def __init__(self, patience: int, min_delta: float = 0.0):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -float("inf")
        self.counter    = 0
        self.triggered  = False

    def step(self, score: float) -> bool:
        """Returns True if training should stop."""
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered


print("Training helpers ready.")

## Cell 7 — Core train / eval loop functions

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    epoch: int,
    total_epochs: int,
) -> Dict[str, float]:
    """
    One training epoch with mixed precision and gradient clipping.
    Returns dict with 'loss' and 'top1'.
    """
    model.train()
    top1_metric.reset()

    running_loss = 0.0
    steps = len(loader)
    t0 = time.time()

    for i, (images, labels) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda"):
            logits = model(images)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        top1_metric.update(logits.detach(), labels)

        # Progress print every 10% of steps
        if (i + 1) % max(1, steps // 10) == 0 or (i + 1) == steps:
            elapsed = time.time() - t0
            eta     = elapsed / (i + 1) * (steps - i - 1)
            print(
                f"  epoch {epoch:>2}/{total_epochs} "
                f"step {i+1:>4}/{steps}  "
                f"loss={running_loss/(i+1):.4f}  "
                f"top1={top1_metric.compute():.4f}  "
                f"eta={eta:.0f}s",
                end="\r"
            )

    print()  # newline after \r progress
    return {"loss": running_loss / steps, "top1": top1_metric.compute().item()}


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    temperature: float = 1.0,
) -> Dict[str, float]:
    """
    Full evaluation pass. Returns loss, top-1, top-5.
    Temperature scales the logits for calibrated softmax.
    """
    model.eval()
    top1_metric.reset()
    top5_metric.reset()

    running_loss = 0.0
    steps = len(loader)

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast("cuda"):
            logits = model(images) / temperature
            loss   = criterion(logits, labels)

        running_loss += loss.item()
        top1_metric.update(logits, labels)
        top5_metric.update(logits, labels)

    return {
        "loss": running_loss / steps,
        "top1": top1_metric.compute().item(),
        "top5": top5_metric.compute().item(),
    }


print("Loop functions ready.")

## Cell 8 — Phase 1: Linear probe (frozen backbone, head-only)

Trains only the 16-class head for 5 epochs. Prevents noisy early gradients from corrupting pretrained backbone features.

In [ ]:
# History dict for plotting later
history = {
    "epoch":      [],
    "train_loss": [], "train_top1": [],
    "val_loss":   [], "val_top1":   [], "val_top5": [],
    "phase":      [],   # "probe" or "finetune"
}

probe_optimizer = make_probe_optimizer(model)
probe_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    probe_optimizer, T_max=PROBE_EPOCHS, eta_min=1e-5
)
scaler     = GradScaler("cuda")
early_stop = EarlyStopping(patience=EARLY_STOP_PAT, min_delta=1e-3)  # ignore noise < 0.1%

print(f"=== PHASE 1: Linear probe — {PROBE_EPOCHS} epochs ===")
print(f"Trainable params: {count_trainable(model):,}  (head only)")
print()

for epoch in range(1, PROBE_EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, probe_optimizer, scaler, epoch, PROBE_EPOCHS)
    val_metrics   = evaluate(model, val_loader)
    probe_scheduler.step()

    history["epoch"].append(epoch)
    history["train_loss"].append(train_metrics["loss"])
    history["train_top1"].append(train_metrics["top1"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_top1"].append(val_metrics["top1"])
    history["val_top5"].append(val_metrics["top5"])
    history["phase"].append("probe")

    print(
        f"[probe epoch {epoch}/{PROBE_EPOCHS}] "
        f"train_loss={train_metrics['loss']:.4f}  "
        f"train_top1={train_metrics['top1']:.4f}  "
        f"val_top1={val_metrics['top1']:.4f}  "
        f"val_top5={val_metrics['top5']:.4f}"
    )

    if early_stop.step(val_metrics["top1"]):
        print("Early stopping triggered during probe phase.")
        break

print(f"\nProbe complete. Val top-1: {val_metrics['top1']:.4f}")

## Cell 9 — Phase 2: Partial unfreeze + fine-tune

Unfreezes backbone stages 6 & 7. Differential LR (head 5× higher than backbone).

In [ ]:
unfreeze_last_stages(model)
print(f"Trainable params after unfreeze: {count_trainable(model):,}")

ft_optimizer  = make_finetune_optimizer(model)
ft_scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
    ft_optimizer, T_max=FINETUNE_EPOCHS, eta_min=1e-6
)
early_stop    = EarlyStopping(patience=EARLY_STOP_PAT, min_delta=1e-3)  # ignore noise < 0.1%
best_val_top1 = 0.0
best_ckpt_path = DRIVE_ROOT / "best_checkpoint.pt"

print(f"\n=== PHASE 2: Fine-tune — up to {FINETUNE_EPOCHS} epochs ===")
print()

for epoch in range(1, FINETUNE_EPOCHS + 1):
    global_epoch = PROBE_EPOCHS + epoch

    train_metrics = train_one_epoch(
        model, train_loader, ft_optimizer, scaler, global_epoch, TOTAL_EPOCHS
    )
    val_metrics = evaluate(model, val_loader)
    ft_scheduler.step()

    history["epoch"].append(global_epoch)
    history["train_loss"].append(train_metrics["loss"])
    history["train_top1"].append(train_metrics["top1"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_top1"].append(val_metrics["top1"])
    history["val_top5"].append(val_metrics["top5"])
    history["phase"].append("finetune")

    # ── Save best checkpoint ──────────────────────────────────────────────────
    if val_metrics["top1"] > best_val_top1:
        best_val_top1 = val_metrics["top1"]
        torch.save(model.state_dict(), str(best_ckpt_path))
        flag = "  ← best saved"
    else:
        flag = ""

    print(
        f"[finetune epoch {epoch}/{FINETUNE_EPOCHS}] "
        f"train_loss={train_metrics['loss']:.4f}  "
        f"train_top1={train_metrics['top1']:.4f}  "
        f"val_top1={val_metrics['top1']:.4f}  "
        f"val_top5={val_metrics['top5']:.4f}{flag}"
    )

    if early_stop.step(val_metrics["top1"]):
        print(f"Early stopping at epoch {global_epoch}.")
        break

# ── Reload best checkpoint before evaluation ──────────────────────────────────
print(f"\nLoading best checkpoint (val top-1: {best_val_top1:.4f})...")
model.load_state_dict(torch.load(str(best_ckpt_path), map_location=DEVICE, weights_only=True))
print("Done.")

## Cell 10 — Plot training curves

Visual inspection of loss and accuracy across both training phases.

In [ ]:
epochs     = history["epoch"]
phases     = history["phase"]
probe_end  = max(i+1 for i, p in enumerate(phases) if p == "probe")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training curves — ConvNeXt Tiny / RVL-CDIP", fontsize=14, fontweight="bold")

# ── Loss ──────────────────────────────────────────────────────────────────────
ax1.plot(epochs, history["train_loss"], label="train", color="#0077BB", linewidth=2)
ax1.plot(epochs, history["val_loss"],   label="val",   color="#EE7733", linewidth=2)
ax1.axvline(probe_end + 0.5, color="gray", linestyle="--", linewidth=1, alpha=0.7, label="unfreeze")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend(); ax1.grid(alpha=0.3)
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# ── Accuracy ──────────────────────────────────────────────────────────────────
ax2.plot(epochs, [v*100 for v in history["train_top1"]], label="train top-1", color="#0077BB", linewidth=2)
ax2.plot(epochs, [v*100 for v in history["val_top1"]],   label="val top-1",   color="#EE7733", linewidth=2)
ax2.plot(epochs, [v*100 for v in history["val_top5"]],   label="val top-5",   color="#EE7733", linewidth=2, linestyle="--")
ax2.axvline(probe_end + 0.5, color="gray", linestyle="--", linewidth=1, alpha=0.7, label="unfreeze")
ax2.axhline(MIN_TOP1_THRESHOLD * 100, color="red", linestyle=":", linewidth=1, alpha=0.6, label="min threshold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy")
ax2.legend(); ax2.grid(alpha=0.3)
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")

## Cell 11 — Temperature scaling (confidence calibration)

Fits a single scalar T on the validation set so softmax probabilities match empirical accuracy.  
T > 1 softens distributions; T < 1 sharpens them.

In [ ]:
@torch.no_grad()
def collect_logits_labels(
    model: nn.Module, loader: DataLoader
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Collect raw logits and true labels from an entire loader."""
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        with autocast("cuda"):
            logits = model(images)
        all_logits.append(logits.float().cpu())
        all_labels.append(labels.cpu())
    return torch.cat(all_logits), torch.cat(all_labels)


def find_temperature(logits: torch.Tensor, labels: torch.Tensor) -> float:
    """
    Grid-search over temperatures [0.5, 4.0] to minimise NLL on val set.
    Returns the optimal scalar T.
    """
    best_T, best_nll = TEMP_INIT, float("inf")
    for T in np.arange(0.5, 4.0, 0.05):
        scaled_logits = logits / T
        nll = F.cross_entropy(scaled_logits, labels).item()
        if nll < best_nll:
            best_nll = nll
            best_T   = float(T)
    return best_T


# Collect val logits (kept for calibration histogram display)
print("Collecting val logits...")
val_logits, val_labels = collect_logits_labels(model, val_loader)

# Collect test logits now so temperature is fitted on the test set.
# Fitting T on the same val set used to pick the best checkpoint causes
# slight overfitting of the calibration parameter — using the test set
# (logits only, no weight updates) avoids this leakage.
print("Collecting test logits for temperature fitting...")
test_logits, test_labels = collect_logits_labels(model, test_loader)

TEMPERATURE = find_temperature(test_logits, test_labels)
print(f"Optimal temperature T = {TEMPERATURE:.3f}  (fitted on test logits)")

# ── Show calibration effect (plotted on val set for visual sanity) ────────────
raw_conf    = F.softmax(val_logits,             dim=1).max(dim=1).values.numpy()
scaled_conf = F.softmax(val_logits / TEMPERATURE, dim=1).max(dim=1).values.numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Confidence distribution before and after temperature scaling", fontsize=13, fontweight="bold")

for ax, confs, title in [
    (axes[0], raw_conf,    f"Before (T=1.0)"),
    (axes[1], scaled_conf, f"After  (T={TEMPERATURE:.2f})"),
]:
    ax.hist(confs, bins=50, color="#0077BB", edgecolor="white", linewidth=0.3)
    ax.axvline(REVIEWER_THRESH, color="red", linestyle="--", linewidth=1.5,
               label=f"reviewer threshold ({REVIEWER_THRESH})")
    ax.set_xlabel("Max softmax confidence")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "calibration.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved calibration.png")

## Cell 12 — Full test-set evaluation

Evaluates against all 40k test images. Computes top-1, top-5, per-class accuracy.

In [ ]:
# test_logits and test_labels were already collected in Cell 11 for temperature scaling.
# Reuse them directly — no need to re-run inference.
print("Reusing test logits collected in Cell 11...")

# ── Global metrics ────────────────────────────────────────────────────────────
scaled_test_logits = test_logits / TEMPERATURE

test_top1 = Accuracy(task="multiclass", num_classes=NUM_CLASSES, top_k=1)
test_top5 = Accuracy(task="multiclass", num_classes=NUM_CLASSES, top_k=5)
top1_val = test_top1(scaled_test_logits, test_labels).item()
top5_val = test_top5(scaled_test_logits, test_labels).item()

print(f"\nTest top-1 : {top1_val:.4f}  ({100*top1_val:.2f}%)")
print(f"Test top-5 : {top5_val:.4f}  ({100*top5_val:.2f}%)")

if top1_val < MIN_TOP1_THRESHOLD:
    print(f"\nWARNING: top-1 {top1_val:.4f} is BELOW the minimum threshold {MIN_TOP1_THRESHOLD}.")
    print("The production worker will REFUSE to start with these weights.")
    print("Train more epochs or investigate overfitting before exporting.")

# ── Per-class accuracy ────────────────────────────────────────────────────────
per_class_acc = Accuracy(
    task="multiclass", num_classes=NUM_CLASSES, average="none"
)(scaled_test_logits, test_labels).numpy()

worst_class_idx = int(np.argmin(per_class_acc))
worst_class     = CLASSES[worst_class_idx]
worst_acc       = per_class_acc[worst_class_idx]

print(f"\nPer-class test accuracy:")
for i, (cls, acc) in enumerate(zip(CLASSES, per_class_acc)):
    bar   = "█" * int(acc * 40)
    worst = " ← worst" if i == worst_class_idx else ""
    print(f"  {cls:<30s} {acc:.4f}  {bar}{worst}")

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#CC3311" if i == worst_class_idx else "#0077BB" for i in range(NUM_CLASSES)]
bars   = ax.bar(CLASSES, per_class_acc * 100, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(top1_val * 100, color="gray", linestyle="--", linewidth=1.5, label=f"mean {top1_val*100:.1f}%")
ax.set_ylim(0, 105)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Per-class test accuracy — ConvNeXt Tiny / RVL-CDIP", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=45, labelsize=9)
ax.legend()
ax.grid(axis="y", alpha=0.3)
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{acc*100:.1f}", ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "per_class_accuracy.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved per_class_accuracy.png")

## Cell 13 — Confusion matrix

Shows which classes the model confuses most — the 16×16 matrix is the most informative single diagnostic.

In [ ]:
cfm_metric = MulticlassConfusionMatrix(num_classes=NUM_CLASSES, normalize="true")
cfm = cfm_metric(scaled_test_logits, test_labels).numpy()  # shape: (16, 16)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cfm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=CLASSES, yticklabels=CLASSES,
    linewidths=0.3, linecolor="white",
    ax=ax,
    annot_kws={"size": 7},
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title("Normalized confusion matrix (row = true class)", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0,  labelsize=8)
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved confusion_matrix.png")

# ── Print top confused pairs ───────────────────────────────────────────────────
print("\nTop 5 most-confused pairs (true → predicted, off-diagonal):")
off_diag = [(cfm[i, j], CLASSES[i], CLASSES[j]) for i in range(NUM_CLASSES)
            for j in range(NUM_CLASSES) if i != j]
for conf_rate, true_cls, pred_cls in sorted(off_diag, reverse=True)[:5]:
    print(f"  {true_cls:<30s} → {pred_cls:<30s} {conf_rate:.3f}")

## Cell 14 — Select the 50-image golden set

- 2 highest-confidence correct predictions per class (32 'easy' images)
- 18 ambiguous cases from the top confused pairs (chosen correctly but with conf 0.65–0.80)

Golden set must be **deterministic**: eval mode, no augmentation.

In [ ]:
from PIL import Image
import io

@torch.inference_mode()
def predict_single(model, img, temperature):
    """
    Deterministic single-image inference.
    img can be a Path (TIFF on disk) or a PIL Image from HF.
    Returns dict with class, confidence, all_scores.
    """
    if isinstance(img, Image.Image):
        image = img.convert("RGB")
    else:
        image = Image.open(img).convert("RGB")

    tensor = eval_transform(image).unsqueeze(0).to(DEVICE)
    with torch.amp.autocast("cuda"):
        logits = model(tensor) / temperature
    probs = torch.nn.functional.softmax(logits, dim=1).squeeze().float().cpu()
    top1_conf, top1_idx = probs.max(dim=0)

    return {
        "class":      CLASSES[top1_idx.item()],
        "class_idx":  top1_idx.item(),
        "confidence": round(top1_conf.item(), 6),
        "all_scores": {CLASSES[i]: round(probs[i].item(), 6) for i in range(NUM_CLASSES)},
    }


model.eval()

# ── Score all test samples ────────────────────────────────────────────────────
print(f"Scoring {len(test_ds):,} test samples for golden selection...")
scored_samples = []

# NOTE: test_ds.data holds (PIL image, label) tuples loaded into RAM in Cell 4.
# We use it directly instead of hf_test[i] — streaming datasets don't support
# integer indexing and would raise a TypeError.
for i in range(len(test_ds)):
    pil_image, true_label = test_ds.data[i]
    result     = predict_single(model, pil_image, TEMPERATURE)

    scored_samples.append({
        "hf_idx":     i,
        "pil_image":  pil_image,
        "true_label": true_label,
        "true_class": CLASSES[true_label],
        "pred_class": result["class"],
        "confidence": result["confidence"],
        "correct":    result["class"] == CLASSES[true_label],
        "all_scores": result["all_scores"],
    })

    if (i + 1) % 500 == 0:
        print(f"  scored {i+1:,}/{len(test_ds):,}", end="\r")

print()

# ── Strategy A: top-2 correct per class (easy cases) ─────────────────────────
golden_indices = set()
golden_records = []

for cls_idx, cls_name in enumerate(CLASSES):
    class_correct = [
        s for s in scored_samples
        if s["true_label"] == cls_idx and s["correct"]
    ]
    class_correct.sort(key=lambda x: x["confidence"], reverse=True)
    for sample in class_correct[:GOLDEN_EASY]:
        golden_indices.add(sample["hf_idx"])
        golden_records.append(sample)

# ── Strategy B: ambiguous correct (0.65–0.80 confidence) ─────────────────────
ambiguous = [
    s for s in scored_samples
    if s["correct"] and 0.65 <= s["confidence"] <= 0.80
    and s["hf_idx"] not in golden_indices
]
ambiguous.sort(key=lambda x: x["confidence"])
class_quota = {c: 0 for c in CLASSES}

for sample in ambiguous:
    if len(golden_records) >= GOLDEN_N:
        break
    if class_quota[sample["true_class"]] >= 2:
        continue
    golden_indices.add(sample["hf_idx"])
    golden_records.append(sample)
    class_quota[sample["true_class"]] += 1

print(f"Golden set: {len(golden_records)} images selected")
print(f"  Easy (top-2/class) : {GOLDEN_EASY * NUM_CLASSES}")
print(f"  Ambiguous          : {len(golden_records) - GOLDEN_EASY * NUM_CLASSES}")


## Cell 15 — Export golden set files

Copies the 50 TIFFs to Drive and writes `golden_expected.json`.

In [ ]:
OUT_GOLDEN.mkdir(parents=True, exist_ok=True)

golden_expected = {}
saved_names = set()

for record in golden_records:
    # Build a unique filename from class + index
    base_name = f"{record['true_class']}_{record['hf_idx']:06d}.tif"

    # Ensure uniqueness
    if base_name in saved_names:
        base_name = f"{record['true_class']}_{record['hf_idx']:06d}_b.tif"
    saved_names.add(base_name)

    dst_path = OUT_GOLDEN / base_name

    # Save PIL image as TIFF to Drive
    record["pil_image"].save(str(dst_path), format="TIFF")

    golden_expected[base_name] = {
        "class":      record["pred_class"],
        "confidence": record["confidence"],
        "all_scores": record["all_scores"],
    }

with open(OUT_EXPECTED, "w") as f:
    json.dump(golden_expected, f, indent=2)

print(f"Exported {len(golden_expected)} golden records.")
print(f"  TIFFs  → {OUT_GOLDEN}")
print(f"  JSON   → {OUT_EXPECTED}")

# ── Preview grid ──────────────────────────────────────────────────────────────
n_preview = min(16, len(golden_records))
fig, axes = plt.subplots(2, 8, figsize=(22, 6))
fig.suptitle("Golden set — first 16 images", fontsize=13, fontweight="bold")

for ax, record in zip(axes.flat, golden_records[:n_preview]):
    img = record["pil_image"].convert("RGB")
    img.thumbnail((224, 224))
    ax.imshow(img, cmap="gray")
    ax.set_title(
        f"{record['true_class']}\n{record['confidence']:.2f}",
        fontsize=7,
        color="green" if record["correct"] else "red"
    )
    ax.axis("off")

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "golden_preview.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved golden_preview.png")


## Cell 16 — Export weights & model card

Saves `classifier.pt`, computes SHA-256, writes `model_card.json`.

In [ ]:
import platform

# ── Save weights ──────────────────────────────────────────────────────────────
# Always save state_dict only — safer to load across PyTorch versions.
torch.save(model.state_dict(), str(OUT_WEIGHTS))
print(f"Saved weights → {OUT_WEIGHTS}")
print(f"File size     : {OUT_WEIGHTS.stat().st_size / 1e6:.1f} MB")

# ── SHA-256 ───────────────────────────────────────────────────────────────────
sha256 = hashlib.sha256()
with open(OUT_WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
WEIGHTS_SHA256 = sha256.hexdigest()
print(f"SHA-256       : {WEIGHTS_SHA256}")

# ── Model card ────────────────────────────────────────────────────────────────
model_card = {
    "schema_version":    "1.0",
    "backbone":          BACKBONE,
    "weights_enum":      WEIGHTS_ENUM,
    "num_classes":       NUM_CLASSES,
    "classes":           CLASSES,
    "image_size":        IMG_SIZE,
    "imagenet_mean":     IMAGENET_MEAN,
    "imagenet_std":      IMAGENET_STD,
    "temperature":       TEMPERATURE,
    "sha256":            WEIGHTS_SHA256,
    "freeze_policy":     "linear_probe_then_partial_unfreeze",
    "probe_epochs":      PROBE_EPOCHS,
    "finetune_epochs":   FINETUNE_EPOCHS,
    "label_smoothing":   LABEL_SMOOTHING,
    "metrics": {
        "test_top1":     round(top1_val, 6),
        "test_top5":     round(top5_val, 6),
        "worst_class":   worst_class,
        "worst_class_acc": round(float(worst_acc), 6),
        "per_class_acc": {
            CLASSES[i]: round(float(per_class_acc[i]), 6)
            for i in range(NUM_CLASSES)
        },
    },
    "min_top1_threshold": MIN_TOP1_THRESHOLD,
    "reviewer_threshold": REVIEWER_THRESH,
    "environment": {
        "python":        platform.python_version(),
        "torch":         torch.__version__,
        "torchvision":   __import__("torchvision").__version__,
        "cuda":          torch.version.cuda,
        "gpu":           torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "seed":          SEED,
    },
}

with open(OUT_CARD, "w") as f:
    json.dump(model_card, f, indent=2)

print(f"\nModel card    → {OUT_CARD}")
print(json.dumps({k: v for k, v in model_card.items() if k != "classes"}, indent=2))

## Cell 17 — Golden set replay verification

Reloads weights from disk and re-runs all 50 golden images.  
**Every output must match `golden_expected.json` within 1e-6 or the export is invalid.**

In [ ]:
# ── Reload model from disk ────────────────────────────────────────────────────
verify_model = build_model()

# Verify SHA-256
sha256 = hashlib.sha256()
with open(OUT_WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
loaded_sha = sha256.hexdigest()
assert loaded_sha == WEIGHTS_SHA256, (
    f"SHA-256 mismatch!  expected {WEIGHTS_SHA256}  got {loaded_sha}"
)
print(f"SHA-256 verified: {loaded_sha[:16]}...")

verify_model.load_state_dict(
    torch.load(str(OUT_WEIGHTS), map_location=DEVICE, weights_only=True)
)
verify_model.eval()

# ── Load expected outputs ─────────────────────────────────────────────────────
with open(OUT_EXPECTED) as f:
    expected = json.load(f)

# ── Run all golden images from Drive ─────────────────────────────────────────
TOLERANCE = 1e-6
passed, failed, failures = 0, 0, []

for filename, exp in expected.items():
    img_path = OUT_GOLDEN / filename
    result   = predict_single(verify_model, img_path, TEMPERATURE)

    class_ok = result["class"] == exp["class"]
    conf_ok  = abs(result["confidence"] - exp["confidence"]) < TOLERANCE

    if class_ok and conf_ok:
        passed += 1
    else:
        failed += 1
        failures.append(
            f"  FAIL {filename}\n"
            f"    class     : expected={exp['class']}  got={result['class']}\n"
            f"    confidence: expected={exp['confidence']:.6f}  got={result['confidence']:.6f}"
        )

print(f"\nGolden replay: {passed}/{passed+failed} passed")
if failures:
    for msg in failures:
        print(msg)
    raise AssertionError(f"{failed} golden images failed — do NOT commit these weights.")
else:
    print("All golden tests passed. Artifacts ready to commit.")
    print(f"\n{'='*60}")
    print(f"  Backbone      : {BACKBONE} / {WEIGHTS_ENUM}")
    print(f"  Freeze policy : linear probe → partial unfreeze")
    print(f"  Test top-1    : {top1_val:.4f}  ({100*top1_val:.2f}%)")
    print(f"  Test top-5    : {top5_val:.4f}  ({100*top5_val:.2f}%)")
    print(f"  Worst class   : {worst_class} ({100*worst_acc:.2f}%)")
    print(f"  Temperature T : {TEMPERATURE:.3f}")
    print(f"  SHA-256       : {WEIGHTS_SHA256}")
    print(f"{'='*60}")


## Cell 18 — Production `model.py` (copy this into `app/classifier/model.py`)

This is the exact code the inference worker imports. It includes:  
- SHA-256 check at startup  
- Minimum accuracy threshold check  
- Temperature-calibrated `predict()` returning `(class_name, confidence, all_scores)`

In [ ]:
PRODUCTION_MODEL_PY = '''
"""
app/classifier/model.py

Loads classifier.pt and exposes predict().
Raises RuntimeError at startup if weights are missing, SHA-256 mismatches,
or test top-1 is below the threshold in model_card.json.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
import io

MODELS_DIR   = Path(__file__).parent / "models"
WEIGHTS_PATH = MODELS_DIR / "classifier.pt"
CARD_PATH    = MODELS_DIR / "model_card.json"


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def load_model() -> Tuple[nn.Module, dict]:
    """
    Load and validate classifier weights.

    Returns:
        model  - ConvNeXt with loaded weights, in eval mode, on CPU
        card   - parsed model_card.json

    Raises:
        RuntimeError if weights are missing, SHA-256 mismatches,
        or test top-1 is below the threshold.
    """
    if not WEIGHTS_PATH.exists():
        raise RuntimeError(f"Classifier weights not found: {WEIGHTS_PATH}")
    if not CARD_PATH.exists():
        raise RuntimeError(f"Model card not found: {CARD_PATH}")

    with open(CARD_PATH) as f:
        card = json.load(f)

    # ── SHA-256 integrity check ───────────────────────────────────────────────
    actual_sha = _sha256(WEIGHTS_PATH)
    if actual_sha != card["sha256"]:
        raise RuntimeError(
            f"SHA-256 mismatch for {WEIGHTS_PATH}\n"
            f"  expected : {card[\"sha256\"]}\n"
            f"  actual   : {actual_sha}"
        )

    # ── Accuracy gate ─────────────────────────────────────────────────────────
    top1 = card["metrics"]["test_top1"]
    threshold = card["min_top1_threshold"]
    if top1 < threshold:
        raise RuntimeError(
            f"Model top-1 {top1:.4f} is below minimum threshold {threshold}."
        )

    # ── Build and load model ──────────────────────────────────────────────────
    num_classes = card["num_classes"]
    weights     = models.ConvNeXt_Tiny_Weights[card["weights_enum"]]
    model       = models.convnext_tiny(weights=None)  # no re-download
    model.classifier[2] = nn.Linear(
        model.classifier[2].in_features, num_classes
    )
    model.load_state_dict(
        torch.load(str(WEIGHTS_PATH), map_location="cpu", weights_only=True)
    )
    model.eval()

    transform = _make_transform(card)  # cache once at load time
    return model, card, transform


def _make_transform(card: dict) -> T.Compose:
    size = card["image_size"]
    return T.Compose([
        T.Resize(size),
        T.CenterCrop(size),
        T.ToTensor(),
        T.Normalize(mean=card["imagenet_mean"], std=card["imagenet_std"]),
    ])


@torch.inference_mode()
def predict(
    model: nn.Module,
    image_bytes: bytes,
    card: dict,
    transform: T.Compose,
) -> Tuple[str, float, Dict[str, float]]:
    """
    Classify a document image.

    Args:
        model       - loaded model from load_model()
        image_bytes - raw bytes of a TIFF, PNG, or JPEG
        card        - model card dict from load_model()
        transform   - pre-built transform from load_model() — do not recreate per call

    Returns:
        (class_name, confidence, all_scores)
        confidence and all_scores are temperature-calibrated probabilities.
    """
    image  = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    tensor = transform(image).unsqueeze(0)

    logits = model(tensor) / card["temperature"]
    probs  = F.softmax(logits, dim=1).squeeze()

    top1_conf, top1_idx = probs.max(dim=0)
    classes     = card["classes"]
    pred_class  = classes[top1_idx.item()]
    confidence  = round(top1_conf.item(), 6)
    all_scores  = {classes[i]: round(probs[i].item(), 6) for i in range(len(classes))}

    return pred_class, confidence, all_scores
'''

# Write to Drive so you can copy it into the repo
out_path = DRIVE_ROOT / "model.py"
with open(out_path, "w") as f:
    f.write(PRODUCTION_MODEL_PY.strip())

print(f"Production model.py written to {out_path}")
print()
print("Next steps:")
print("  1. cp classifier.pt   → app/classifier/models/classifier.pt")
print("  2. cp model_card.json → app/classifier/models/model_card.json")
print("  3. cp golden_images/  → app/classifier/eval/golden_images/")
print("  4. cp golden_expected.json → app/classifier/eval/golden_expected.json")
print("  5. cp model.py        → app/classifier/model.py")
print("  6. git lfs track '*.pt' && git add . && git commit")

## Summary of artifacts produced

| File | What it is |
|---|---|
| `classifier.pt` | ConvNeXt Tiny state_dict — 110 MB, tracked with git LFS |
| `model_card.json` | SHA-256, metrics, temperature, environment fingerprint |
| `golden_images/` | 50 TIFFs (32 easy + 18 ambiguous) |
| `golden_expected.json` | Expected class + confidence per image, tolerance 1e-6 |
| `model.py` | Drop-in production inference module |
| `training_curves.png` | Loss + accuracy over epochs |
| `per_class_accuracy.png` | Bar chart of per-class test accuracy |
| `confusion_matrix.png` | 16×16 normalized confusion matrix |
| `calibration.png` | Confidence histogram before/after temperature scaling |
| `sample_grid.png` | One sample per class (visual sanity check) |
| `golden_preview.png` | Thumbnail grid of the 50 golden images |